<a href="https://colab.research.google.com/github/Ananyasaini06/Small-Object-Detection/blob/main/Small_Object_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Set up the T4 runtime

Select a T4 GPU before running this cell. It installs the packages and loads the project code.

In [1]:
%pip install -q "ultralytics>=8.3,<9" "sahi>=0.11.20,<0.12" "gradio>=5,<6"
from pathlib import Path
import subprocess, sys, torch
from ultralytics import YOLO
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → T4 GPU'
repo = Path('/content/Small-Object-Detection')
if not repo.exists():
    subprocess.run(['git','clone','--branch','main','https://github.com/Ananyasaini06/Small-Object-Detection.git',str(repo)], check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only','origin','main'], check=True)
sys.path.insert(0, str(repo))
from datetime import datetime, timezone
def run_id():
    return datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
from comparison import benchmark
from demo import launch


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.14.0 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.52.1 which is incompatible.
python-fasthtml 0.14.18 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.
google-genai 2.12.1 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 2.12.3 which is incompatible.
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


## 2. Mount Drive and find YOLO weights

Check the recovered path before training. If your checkpoint is in another folder, enter its path in manual.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive')
project = drive_root / 'Small-Object-Detection'
project.mkdir(exist_ok=True)
# Set this to your own trusted VisDrone best.pt or last.pt if stored elsewhere.
manual = None
candidates = [Path(manual)] if manual else []
if manual and not candidates[0].is_file():
    raise FileNotFoundError(manual)
for folder in (project, drive_root/'P10_runs', Path('/content/P10_runs'), repo):
    for filename in ('best.pt', 'last.pt'):
        candidates.extend(sorted(folder.glob('**/final_baseline*/weights/'+filename), key=lambda p:p.stat().st_mtime, reverse=True))
        candidates.extend([folder/filename])
weights = next((p for p in candidates if p.is_file()), None)
if weights:
    detector = YOLO(str(weights))
    assert len(detector.names) == 10, 'Verify this is a trained VisDrone checkpoint'
print('Recovered:', weights)

## 3. Prepare VisDrone

Run this cell for training or evaluation. A demo using saved weights can skip it.

In [ ]:
from ultralytics.data.utils import check_det_dataset, img2label_paths
data = check_det_dataset('VisDrone.yaml')
val_dir = Path(data['val'])
label_dir = Path(img2label_paths([str(val_dir / 'example.jpg')])[0]).parent
assert val_dir.is_dir() and label_dir.is_dir()


## 4. Optional training if recovery found no weights

With Recovered: None, set TRAIN=True after checking your Drive backups. Leave RESUME=False for a new 15-epoch run. The cell checks Drive before training and verifies both files afterward.

In [ ]:
trained = False
TRAIN = False
RESUME = False
if RESUME or (TRAIN and weights is None):
    import os, tempfile
    if not os.path.ismount('/content/drive') or not project.resolve().is_relative_to(drive_root.resolve()):
        raise RuntimeError('Mount Google Drive in section 2 before training.')
    with tempfile.NamedTemporaryFile(dir=project) as probe:
        probe.write(b'Drive checkpoint test')
        probe.flush()
        os.fsync(probe.fileno())
if RESUME:
    assert weights, 'Recover a trusted checkpoint in section 2 first.'
    last = weights.with_name('last.pt')
    assert last.is_file(), 'Select the interrupted run in section 2.'
    model = YOLO(str(last))
    assert model.ckpt.get('optimizer') and model.ckpt.get('epoch', -1) >= 0, 'This checkpoint is finalized and cannot resume.'
    saved_dir = Path(model.ckpt.get('train_args', {}).get('save_dir') or last.parent.parent)
    if not saved_dir.is_absolute() or not saved_dir.resolve().is_relative_to(project.resolve()) or saved_dir.resolve() != last.parent.parent.resolve():
        raise ValueError('Resume the original Drive run with its original save directory.')
    model.train(resume=True, device=0)
    trained = True
elif TRAIN and weights is None:
    model = YOLO('yolo11n.pt')
    model.train(data='VisDrone.yaml', epochs=15, imgsz=640, batch=8, device=0,
                workers=2, project=str(project), name='final_baseline', seed=42, patience=0, save=True)
    trained = True
if trained:
    for checkpoint in (Path(model.trainer.best), Path(model.trainer.last)):
        if not checkpoint.resolve().is_relative_to(project.resolve()) or not checkpoint.is_file() or not checkpoint.stat().st_size:
            raise RuntimeError(f'Missing Drive checkpoint: {checkpoint}')
        print('Saved:', checkpoint)
    weights = Path(model.trainer.best)
print('Inference checkpoint:', weights)


## 5. Evaluate models and optionally repeat the archived experiments

The default compares three models. Enable the tile study for latency, counts, ten-class small AP50, both graphs and sample panels. Enable full validation for YOLO mAP50 and mAP50–95. Each execution writes new results to Drive.

In [ ]:
RUN_EVALUATION = True
REPEAT_HISTORICAL_PROTOCOL = False
RUN_TILE_STUDY = False
RUN_FULL_VALIDATION = False
if any((RUN_EVALUATION, REPEAT_HISTORICAL_PROTOCOL, RUN_TILE_STUDY, RUN_FULL_VALIDATION)):
    assert weights and weights.is_file(), 'Recover or train YOLO first.'
    evaluation_dir = project / 'evaluation' / run_id()
    if RUN_EVALUATION:
        display(benchmark(weights, val_dir, label_dir, evaluation_dir))
    evaluation_dir.mkdir(parents=True, exist_ok=True)
    if REPEAT_HISTORICAL_PROTOCOL:
        from evaluate import evaluate
        evaluate(str(weights), val_dir, label_dir, evaluation_dir)
    print('Saved to', evaluation_dir)

    if RUN_FULL_VALIDATION:
        metrics = YOLO(str(weights)).val(data='VisDrone.yaml', imgsz=640, device=0, project=str(evaluation_dir), name='validation')
        print('mAP50:', metrics.box.map50, 'mAP50–95:', metrics.box.map)
    if RUN_TILE_STUDY:
        from tile_experiments import run
        run(weights, val_dir, label_dir, evaluation_dir / 'tiles')


## 6. Compare detections on one image

Upload a JPG or PNG, adjust the confidence threshold and compare the three outputs.

In [ ]:
RUN_DEMO = True
if RUN_DEMO:
    assert weights and weights.is_file(), 'Recover or train YOLO first.'
    launch(weights, project)


## 7. Back up this run

Save the available checkpoints, results and software versions in a dated ZIP on Drive.

In [ ]:
import zipfile
assert weights and weights.is_file(), 'Recover or train YOLO first.'
backup = project / ('submission-' + run_id() + '.zip')
with zipfile.ZipFile(backup, 'x') as z:
    for path in {weights, weights.with_name('best.pt'), weights.with_name('last.pt')}:
        if path.is_file():
            z.write(path, 'weights/' + path.name)
    if 'evaluation_dir' in globals():
        for path in evaluation_dir.rglob('*'):
            if path.is_file():
                z.write(path, 'evaluation/' + str(path.relative_to(evaluation_dir)))
    z.writestr('requirements-runtime.txt', subprocess.check_output([sys.executable,'-m','pip','freeze']))
    z.writestr('repository-commit.txt', subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD']))
print(backup)
